# 1. Import of input data

In [ ]:
import nbimporter
import sys
import os
from pathlib import Path

In [ ]:
#Import libraries
try:
    current_dir = RUN_DIR
    print(f"📂 BiomassBoiler.ipynb executed by RUN.ipynb — using directory: {current_dir}")
except NameError:

    current_dir = Path().resolve()
    print(f"📁 BiomassBoiler.ipynb executed standalone — using directory: {current_dir}")

Libraries_path = (current_dir / "Libraries.ipynb").resolve()

# If not found, try in the parent levels
if not Libraries_path.exists():
    Libraries_path = (current_dir.parent / "Libraries.ipynb").resolve()

# Run the file
if Libraries_path.exists():
    print(f"📘 Importo librerie da: {Libraries_path}")
    get_ipython().run_line_magic('run', f'"{Libraries_path}"')
else:
    print("⚠️ Libraries.ipynb not found! Some functions might not be available.")

In [ ]:
energysystem = pd.read_excel(energysystem_path, sheet_name = 'Biomass boiler')

# 2. Input parameters

In [ ]:
# DASHBORD - VALUE INPUTS
Power = energysystem['Thermal capacity [kW]']
Energy_produced = energysystem['Energy produced [kWh /year]']
Efficiency = energysystem['Efficiency (between 0 and 1)']

In [ ]:
# DASHBOARD - INPUT PARAMETERS
Power = Power

Efficiency_LHV = Efficiency

# DEFAULT PARAMETERS
# Heating values of wood on a DRY basis, consistent with the ecoinvent input
# "market for wood chips, dry, measured as dry mass" (kg of dry wood).
# LHV: net calorific value of oven-dry wood = 19 MJ/kg
#      Forest Research (UK), "Typical calorific values of fuels"
#      https://www.forestresearch.gov.uk/tools-and-resources/biomass-energy-resources/reference-biomass/facts-figures/typical-calorific-values-of-fuels/
#      (consistent with Huhtinen, "Wood properties as a fuel", 5EURES: pine 19.3, birch 19.2, alder 19.0 MJ/kg dry)
# HHV: LHV + latent heat of the water formed by combustion
#      = 19.0 + 9 * 0.061 (H content of dry wood) * 2.442 MJ/kg = 20.3 MJ/kg
LHV = 19.0  # MJ/kg dry wood
HHV = 20.3  # MJ/kg dry wood

Lifespan = np.where(Power <= 100, 15, 20)
RSP = np.array([RSP])
Efficiency_HHV = Efficiency_LHV*(LHV/HHV)  # Efficiency based on HHV

TotEnergyNeeds = np.array([Energy_produced])  # Energy need for 1 year kWh per year

# 3. Life cycle inventories

## 3.1 Module A1-3

In [ ]:
P = Power  # alias

m1 = (P >= 5) & (P <= 20)
m2 = (P > 20) & (P <= 100)
m3 = (P > 100) & (P <= 500)
m4 = (P > 500)

i = np.ceil(P / 100)

# MASS
Mass = np.select(
    [m1, m2, m3, m4],
    [
        176,
        613,
        4138,
        (140.86 * np.log(P / i) - 415) * i
    ]
)

Biomassboiler_HDPE = np.select(
    [m1, m2, m3, m4],
    [0.0011 * Mass, 0.009 * Mass, 0.001 * Mass, 0.001 * Mass]
)

Biomassboiler_alkydpaint = np.select(
    [m1, m2, m3, m4],
    [0, 0.001 * Mass, 0.001 * Mass, 0.001 * Mass]
)

Biomassboiler_aluminium = np.select(
    [m1, m2, m3, m4],
    [0.0110 * Mass, 0, 0.002 * Mass, 0.002 * Mass]
)

Biomassboiler_chromiumsteel = np.select(
    [m1, m2, m3, m4],
    [0, 0.0049 * Mass, 0.007 * Mass, 0.007 * Mass]
)

Biomassboiler_claybrick = np.select(
    [m1, m2, m3, m4],
    [0, 0, 0.166 * Mass, 0.166 * Mass]
)

Biomassboiler_castiron = np.select(
    [m1, m2, m3, m4],
    [0.1766 * Mass, 0, 0, 0]
)

Biomassboiler_ceramictile = np.select(
    [m1, m2, m3, m4],
    [0.1821 * Mass, 0, 0, 0]
)

Biomassboiler_copper = np.select(
    [m1, m2, m3, m4],
    [0.0017 * Mass, 0.0003 * Mass, 0.001 * Mass, 0.001 * Mass]
)

Biomassboiler_electronics = np.select(
    [m1, m2, m3, m4],
    [0.01 * Mass, 0.007 * Mass, 0.001 * Mass, 0.001 * Mass]
)

Biomassboiler_flatglasscoated = np.select(
    [m1, m2, m3, m4],
    [0.0248 * Mass, 0, 0, 0]
)

Biomassboiler_lowalloyedsteel = np.select(
    [m1, m2, m3, m4],
    [0.698 * Mass, 0.754 * Mass, 0.698 * Mass, 0.698 * Mass]
)

Biomassboiler_lubricatingoil = np.select(
    [m1, m2, m3, m4],
    [0.0022 * Mass, 0.001 * Mass, 0.001 * Mass, 0.001 * Mass]
)

Biomassboiler_polystyrene = np.select(
    [m1, m2, m3, m4],
    [0, 0.029 * Mass, 0.013 * Mass, 0.013 * Mass]
)

Biomassboiler_sheetrollingsteel = np.select(
    [m1, m2, m3, m4],
    [0.0635 * Mass, 0.0062 * Mass, 0.044 * Mass, 0.044 * Mass]
)

Biomassboiler_stonewool = np.select(
    [m1, m2, m3, m4],
    [0.064 * Mass, 0.0086 * Mass, 0.064 * Mass, 0.064 * Mass]
)

Biomassboiler_concrete = 4.9171 * np.log(P) - 10.058
Biomassboiler_electricityA13 = 320.61 * np.log(P) - 561.92
Biomassboiler_heat = 569.68 * np.log(P) - 521.23

## 3.2 Module A4

In [ ]:
Biomassboiler_Truk_16_32 = ((500 + 200)*(Mass)/1000)
Biomassboiler_Light_Truk = (50*(Mass)/1000)

## 3.3 Module B2

In [ ]:
Biomassboiler_Light_Truk_B2 = 50 * (Lifespan/2) * 0.01 * (Mass/1000)

## 3.4 Module B4

In [ ]:
BiomassboilerNumberReplacements = (np.ceil(np.divide(ReqSL, Lifespan) - 1))
SFBiomassboiler = (RSP / ReqSL)  # Scaling factor

## 3.5 Module B6

In [ ]:
# ------------------------------------------------------------------
# B6 - Biomass boiler
# Wood input is expressed as dry mass; direct-emission factors in the
# source inventory are expressed per MJ fuel on an HHV basis.
# ------------------------------------------------------------------

# Useful heat delivered over the reference study period [MJ heat]
Biomassboiler_heat_output_MJ = (
    np.asarray(Energy_produced, dtype=float).flatten()
    * RSP
    * 3.6
)

# Fuel energy required at boiler inlet [MJ fuel, HHV]
Biomassboiler_fuel_energy_HHV_MJ = (
    Biomassboiler_heat_output_MJ
    / Efficiency_HHV
)

# Physical biomass input [kg dry wood]
Biomassboiler_wood_chips_input_kg = (
    Biomassboiler_fuel_energy_HHV_MJ
    / HHV
)

# Same physical fuel balance expressed on an LHV basis [MJ fuel, LHV]
Biomassboiler_fuel_energy_LHV_MJ = (
    Biomassboiler_wood_chips_input_kg
    * LHV
)

# Compatibility alias: remove after updating the B6 exchange in RUN
Biomassboiler_Energyvector = Biomassboiler_wood_chips_input_kg


# ------------------------------------------------------------------
# Auxiliary electricity
# 0.0056 kWh per MJ fuel, HHV basis.
# ------------------------------------------------------------------

Biomassboiler_auxiliary_electricity_kWh = (
    0.0056
    * Biomassboiler_fuel_energy_HHV_MJ
)

Biomassboiler_electricityB6 = (
    Biomassboiler_auxiliary_electricity_kWh
    * Grid
)

Biomassboiler_electricityB6PVExt = (
    Biomassboiler_auxiliary_electricity_kWh
    * PV
)

Biomassboiler_electricityB6RECSPV = (
    Biomassboiler_electricityB6
    * RECSPV
)

Biomassboiler_electricityB6RECSHydro = (
    Biomassboiler_electricityB6
    * RECSHydro
)

Biomassboiler_electricityB6RECSEolic = (
    Biomassboiler_electricityB6
    * RECSEolic
)


# ------------------------------------------------------------------
# Direct emissions
# Factors are per MJ fuel input, HHV basis.
# ------------------------------------------------------------------

Biomassboiler_emission_basis_MJ = (
    Biomassboiler_fuel_energy_HHV_MJ
)

Biomassboiler_acetaldehyde = (
    0.000000061126
    * Biomassboiler_emission_basis_MJ
)

Biomassboiler_ammonia = (
    0.0000017335
    * Biomassboiler_emission_basis_MJ
)

Biomassboiler_benzene = (
    0.00000091189
    * Biomassboiler_emission_basis_MJ
)

Biomassboiler_benzoapyrene = (
    0.000000000501039
    * Biomassboiler_emission_basis_MJ
)

Biomassboiler_bromine = (
    0.0000000006012472
    * Biomassboiler_emission_basis_MJ
)

Biomassboiler_cadmium = (
    0.000000000701455
    * Biomassboiler_emission_basis_MJ
)

Biomassboiler_calcium = (
    0.000005862161
    * Biomassboiler_emission_basis_MJ
)

Biomassboiler_carbondioxide_biogenic = (
    (0.00097 + 0.091)
    * Biomassboiler_emission_basis_MJ
)

Biomassboiler_carbonmonoxide_biogenic = (
    0.0
    * Biomassboiler_emission_basis_MJ
)

Biomassboiler_chlorine = 0.0000001803741 * Biomassboiler_emission_basis_MJ
Biomassboiler_chromium = 0.00000000396823 * Biomassboiler_emission_basis_MJ
Biomassboiler_chromiumVI = 0.00000000004008315 * Biomassboiler_emission_basis_MJ
Biomassboiler_copperB6 = 0.00000002204573 * Biomassboiler_emission_basis_MJ
Biomassboiler_dinitrogenmonoxide = 0.000003006236 * Biomassboiler_emission_basis_MJ
Biomassboiler_dioxine = 0.0000000000000310644 * Biomassboiler_emission_basis_MJ
Biomassboiler_fluorine = 0.00000000501039 * Biomassboiler_emission_basis_MJ
Biomassboiler_formaldeide = 0.0000001302702 * Biomassboiler_emission_basis_MJ
Biomassboiler_hydrocarbons_aliphatic_unsaturated = 0.00000000911917 * Biomassboiler_emission_basis_MJ
Biomassboiler_lead = 0.00000310644 * Biomassboiler_emission_basis_MJ
Biomassboiler_magnesium = 0.000000002505109 * Biomassboiler_emission_basis_MJ
Biomassboiler_manganese = 0.00000036074 * Biomassboiler_emission_basis_MJ
Biomassboiler_mercury = 0.00000017035 * Biomassboiler_emission_basis_MJ
Biomassboiler_methanebiogenic = 0.000000000300623 * Biomassboiler_emission_basis_MJ
Biomassboiler_mxylene = 0.0000114236984624604 * Biomassboiler_emission_basis_MJ
Biomassboiler_nickel = 0.000000120249 * Biomassboiler_emission_basis_MJ
Biomassboiler_nitrogenoxides = 0.0000601247287497914 * Biomassboiler_emission_basis_MJ
Biomassboiler_NMVOC = 0.0000266552964124075 * Biomassboiler_emission_basis_MJ
Biomassboiler_PAH = 0.0000000120249 * Biomassboiler_emission_basis_MJ
Biomassboiler_particulates_25 = 0.0000360748372498748 * Biomassboiler_emission_basis_MJ
Biomassboiler_particulates_10 = 0.0000020041576 * Biomassboiler_emission_basis_MJ
Biomassboiler_pentachlorophenol = 0.0000000000081168 * Biomassboiler_emission_basis_MJ
Biomassboiler_posphorus = 0.0000003006 * Biomassboiler_emission_basis_MJ
Biomassboiler_potassium = 0.0000234486442124186 * Biomassboiler_emission_basis_MJ
Biomassboiler_sodium = 0.0000013027 * Biomassboiler_emission_basis_MJ
Biomassboiler_sulfurdioxide = 0.00000250519 * Biomassboiler_emission_basis_MJ
Biomassboiler_toluene = 0.0000003006 * Biomassboiler_emission_basis_MJ
Biomassboiler_water = 0.0000535110085873145 * Biomassboiler_emission_basis_MJ
Biomassboiler_zinc = 0.0000003006 * Biomassboiler_emission_basis_MJ

## 3.6 Module C2

In [ ]:
Biomassboiler_Transport_to_treatment_plant = (50 * (Mass/1000))

## 3.7 Module C3

In [ ]:
Biomassboiler_EOL_HDPE = Biomassboiler_HDPE
Biomassboiler_EOL_alkydpaint = Biomassboiler_alkydpaint
Biomassboiler_EOL_aluminium = Biomassboiler_aluminium
Biomassboiler_EOL_chromiumsteel = Biomassboiler_chromiumsteel
Biomassboiler_EOL_claybrick = Biomassboiler_claybrick
Biomassboiler_EOL_castiron = Biomassboiler_castiron
Biomassboiler_EOL_ceramictile = Biomassboiler_ceramictile
Biomassboiler_EOL_concrete = Biomassboiler_concrete
Biomassboiler_EOL_copper = Biomassboiler_copper
Biomassboiler_EOL_electricity = Biomassboiler_electricityA13
Biomassboiler_EOL_electronics = Biomassboiler_electronics
Biomassboiler_EOL_flatglasscoated = Biomassboiler_flatglasscoated
Biomassboiler_EOL_heat = Biomassboiler_heat
Biomassboiler_EOL_lowalloyedsteel = Biomassboiler_lowalloyedsteel
Biomassboiler_EOL_lubricatingoil = Biomassboiler_lubricatingoil
Biomassboiler_EOL_polystyrene = Biomassboiler_polystyrene
Biomassboiler_EOL_sheetrollingsteel = Biomassboiler_sheetrollingsteel
Biomassboiler_EOL_stonewool = Biomassboiler_stonewool

## 3.8 Module C4

In [ ]:
Biomassboiler_EOL_inc_PEHD = Biomassboiler_EOL_HDPE * IncinerationPEHD
Biomassboiler_EOL_landfill_PEHD = Biomassboiler_EOL_HDPE * LandfillPEHD
Biomassboiler_EOL_inc_alkydpaint = Biomassboiler_EOL_alkydpaint    
Biomassboiler_EOL_landfill_aluminium = Biomassboiler_EOL_aluminium * LandfillMetals
Biomassboiler_EOL_landfill_chromiumsteel = Biomassboiler_EOL_chromiumsteel * LandfillMetals2
Biomassboiler_EOL_landfill_claybrick = Biomassboiler_EOL_claybrick * LandfillClaybrick    
Biomassboiler_EOL_landfill_castiron = Biomassboiler_EOL_castiron * LandfillMetals2
Biomassboiler_EOL_landfill_ceramictile = Biomassboiler_EOL_ceramictile * LandfillClaybrick
Biomassboiler_EOL_landfill_concrete = Biomassboiler_EOL_concrete * LandfillConcrete
Biomassboiler_EOL_landfill_copper = Biomassboiler_EOL_copper * LandfillMetals
Biomassboiler_EOL_inc_electronics = Biomassboiler_EOL_electronics * IncinerationElectronics #kg
Biomassboiler_EOL_landfill_electronics = Biomassboiler_EOL_electronics * LandfillElectronics #kg
Biomassboiler_EOL_landfill_flatglasscoated = Biomassboiler_EOL_flatglasscoated * LandfillGlass #kg
Biomassboiler_EOL_landfill_lowalloyedsteel = Biomassboiler_EOL_lowalloyedsteel * LandfillMetals
Biomassboiler_EOL_inc_lubricatingoil = Biomassboiler_lubricatingoil
Biomassboiler_EOL_inc_polystyrene = Biomassboiler_EOL_polystyrene * IncinerationEPS
Biomassboiler_EOL_landfill_sheetrollingsteel = Biomassboiler_EOL_sheetrollingsteel * LandfillMetals
Biomassboiler_EOL_landfill_stonewool = Biomassboiler_EOL_stonewool * LandfillMineralWool


## 3.9 Module D1 (related to the export of secondary materials)

In [ ]:
Biomassboiler_EOL_recycling_aluminium = (RecyclingMetals * RecyclingEfficiencyMetals * Biomassboiler_EOL_aluminium) - (RecycledContentMetals * Biomassboiler_EOL_aluminium)
Biomassboiler_EOL_primary_aluminium = ((RecycledContentMetals * Biomassboiler_EOL_aluminium) - (RecyclingMetals * RecyclingEfficiencyMetals * Biomassboiler_EOL_aluminium)) * SRmetals

Biomassboiler_EOL_recycling_castiron = (RecyclingMetals * RecyclingEfficiencyMetals * Biomassboiler_EOL_castiron) - (RecycledContentMetals * Biomassboiler_EOL_castiron)
Biomassboiler_EOL_primary_castiron = ((RecycledContentMetals * Biomassboiler_EOL_castiron) - (RecyclingMetals * RecyclingEfficiencyMetals * Biomassboiler_EOL_castiron)) * SRmetals

Biomassboiler_EOL_recycling_copper = (RecyclingMetals * RecyclingEfficiencyMetals * Biomassboiler_EOL_copper) - (RecycledContentMetals * Biomassboiler_EOL_copper)
Biomassboiler_EOL_primary_copper = ((RecycledContentMetals * Biomassboiler_EOL_copper) - (RecyclingMetals * RecyclingEfficiencyMetals * Biomassboiler_EOL_copper)) * SRmetals

Biomassboiler_EOL_recycling_HDPE = (RecyclingPEHD * RecyclingEfficiencyPlastics * Biomassboiler_EOL_HDPE) - (RecycledContentPlastic * Biomassboiler_EOL_HDPE)
Biomassboiler_EOL_primary_HDPE = ((RecycledContentPlastic * Biomassboiler_EOL_HDPE) - (RecyclingPEHD * RecyclingEfficiencyPlastics * Biomassboiler_EOL_HDPE)) * SRinert

Biomassboiler_EOL_recycling_sheetrollingsteel = (RecyclingMetals * RecyclingEfficiencyMetals * Biomassboiler_EOL_sheetrollingsteel) - (RecycledContentMetals * Biomassboiler_EOL_sheetrollingsteel)
Biomassboiler_EOL_primary_sheetrollingsteel = ((RecycledContentMetals * Biomassboiler_EOL_sheetrollingsteel) - (RecyclingMetals * RecyclingEfficiencyMetals * Biomassboiler_EOL_sheetrollingsteel)) * SRmetals

Biomassboiler_EOL_recycling_chromiumsteel = (RecyclingMetals2 * RecyclingEfficiencyMetals * Biomassboiler_EOL_chromiumsteel) - (RecycledContentMetals * Biomassboiler_EOL_chromiumsteel)
Biomassboiler_EOL_primary_chromiumsteel = ((RecycledContentMetals * Biomassboiler_EOL_chromiumsteel) - (RecyclingMetals2 * RecyclingEfficiencyMetals * Biomassboiler_EOL_chromiumsteel)) * SRmetals

Biomassboiler_EOL_recycling_chromiumsteel = (RecyclingMetals2 * RecyclingEfficiencyMetals * Biomassboiler_EOL_chromiumsteel) - (RecycledContentMetals * Biomassboiler_EOL_chromiumsteel)
Biomassboiler_EOL_primary_chromiumsteel = ((RecycledContentMetals * Biomassboiler_EOL_chromiumsteel) - (RecyclingMetals2 * RecyclingEfficiencyMetals * Biomassboiler_EOL_chromiumsteel)) * SRmetals

Biomassboiler_EOL_recycling_claybrick = (RecyclingClaybrick * RecyclingEfficiencyClaybrick * Biomassboiler_EOL_claybrick) - (RecycledContentClaybrick * Biomassboiler_EOL_claybrick)
Biomassboiler_EOL_primary_claybrick = ((RecycledContentClaybrick * Biomassboiler_EOL_claybrick) - (RecyclingClaybrick * RecyclingEfficiencyClaybrick * Biomassboiler_EOL_claybrick)) * SRclaybrick

Biomassboiler_EOL_recycling_lowalloyedsteel = (RecyclingMetals * RecyclingEfficiencyMetals * Biomassboiler_EOL_lowalloyedsteel) - (RecycledContentMetals * Biomassboiler_EOL_lowalloyedsteel)
Biomassboiler_EOL_primary_lowalloyedsteel = ((RecycledContentMetals * Biomassboiler_EOL_lowalloyedsteel) - (RecyclingMetals * RecyclingEfficiencyMetals * Biomassboiler_EOL_lowalloyedsteel)) * SRmetals

Biomassboiler_EOL_recycling_concrete = (RecyclingConcrete * RecyclingEfficiencyConcrete * Biomassboiler_EOL_concrete * 2400) - (RecycledContentConcrete * Biomassboiler_EOL_concrete * 2400)
Biomassboiler_EOL_primary_concrete = ((RecycledContentConcrete * Biomassboiler_EOL_concrete * 2400) - (RecyclingConcrete * RecyclingEfficiencyConcrete * Biomassboiler_EOL_concrete * 2400)) * SRinert

## 3.10 Module D3 (related to the export of energy as a result of waste incineration)

In [ ]:
Biomassboiler_EOL_energyrecoveryheat_alkydpaint = - Biomassboiler_EOL_inc_alkydpaint * LHVElastomer * Xincheat
Biomassboiler_EOL_energyrecoveryele_alkydpaint = (- Biomassboiler_EOL_inc_alkydpaint * LHVElastomer * Xincele) / 3.6
Biomassboiler_EOL_energyrecoveryheat_electronics = - Biomassboiler_EOL_inc_electronics * LHVElectronics * Xincheat
Biomassboiler_EOL_energyrecoveryele_electronics = (- Biomassboiler_EOL_inc_electronics * LHVElectronics * Xincele) / 3.6
Biomassboiler_EOL_energyrecoveryheat_PEHD = - Biomassboiler_EOL_inc_PEHD * LHVHDPE * Xincheat
Biomassboiler_EOL_energyrecoveryele_PEHD = (- Biomassboiler_EOL_inc_PEHD * LHVHDPE * Xincele) / 3.6
Biomassboiler_EOL_energyrecoveryheat_polystyrene = - Biomassboiler_EOL_inc_polystyrene * LHVEPS * Xincheat
Biomassboiler_EOL_energyrecoveryele_polystyrene = (- Biomassboiler_EOL_inc_polystyrene * LHVEPS * Xincele) / 3.6

# 4. Data for export 

In [ ]:
ThermalEnergyProduced_Biomassboiler = Energy_produced * RSP